# Production workflow: pipeline_v4
This is the only recommended production entry point. It processes every user supplied by the external sample, performs QC/home/AGEB, and routes every user whose GPS passes technical QC. Home confidence is retained for later inventory use but never blocks mobility processing. It never samples or replaces users. Each execution creates a unique `Outputs/runs/<run_id>/` directory.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / 'pipeline_v4').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from pipeline_v4.preprocessing import PreprocessingConfig
from pipeline_v4.preprocessing.gps_home_sampling.workflow import HomeConfig
from pipeline_v4.src.run_workflow import run_production, validate_production_environment
PRODUCTION_ENVIRONMENT = validate_production_environment()
print(PRODUCTION_ENVIRONMENT)

## Configuration
Use a supplied GPS Parquet and its matching AGEB file. `SUPPLIED_USER_IDS=None` processes all users in the input. The run directory label is derived automatically from the input filename; uniqueness is automatic. Set `SAVE_PREPROCESSED_GPS=False` for large runs when persistence of that derived intermediate is unnecessary.

In [ ]:
# Input files: use the supplied GPS data and its matching AGEB boundary file.
SUPPLIED_GPS_PATH = ROOT / 'Inputs' / 'GPS User Data' / 'supplied_gps.parquet'
AGEB_PATH = ROOT / 'Inputs' / 'Infrastructure' / 'AGEB' / 'AGEB_ZMM.json'
# User/date filters: None keeps every user/date present in the supplied input.
SUPPLIED_USER_IDS = None
START_DATE = None
END_DATE = None
# Half-open UTC acquisition coverage. Naive timestamps are also interpreted as UTC.
# Set exact boundaries when known; None retains edge days with unknown completeness.
COVERAGE_START_UTC = None
COVERAGE_END_UTC = None
# Processing scope: MAX_USERS keeps the first users in source-file appearance order; None keeps all.
MAX_USERS = None
MAX_DAYS_PER_USER = None
# Residence metadata: use None only for lax candidate assessment.
HOME_MIN_NIGHTS = 3
# Run naming is automatic: Outputs/runs/<timestamp>_<input_filename>/
SAVE_PREPROCESSED_GPS = True
# Runtime: the production default uses two workers.
N_JOBS = 2
# Output format: summary = analysis/visualization; detailed = audit/debugging; both = write both files.
OUTPUT_MODE = 'summary'  # allowed values: 'summary', 'detailed', 'both'
# Progress feedback: show phase messages and a user-day progress bar while the run is active.
SHOW_PROGRESS = True
# Safety switch: review all settings before changing this to True.
EXECUTE_PRODUCTION_RUN = False

## Execute preprocessing + pipeline_v4
Home inference keeps the validated 22:00–05:00 baseline. Routes, modes, emissions, ledger, figures directory, and manifest all belong to the same run.

In [ ]:
result = None
if EXECUTE_PRODUCTION_RUN:
    result = run_production(
        SUPPLIED_GPS_PATH, AGEB_PATH,
        preprocessing_config=PreprocessingConfig(
            start_date=START_DATE, end_date=END_DATE,
            coverage_start=COVERAGE_START_UTC, coverage_end=COVERAGE_END_UTC,
        ),
        home_config=HomeConfig(min_nights=HOME_MIN_NIGHTS), user_ids=SUPPLIED_USER_IDS,
        save_preprocessed_gps=SAVE_PREPROCESSED_GPS, n_jobs=N_JOBS, output_mode=OUTPUT_MODE,
        limit_users=MAX_USERS, limit_days_per_user=MAX_DAYS_PER_USER, show_progress=SHOW_PROGRESS,
    )
    print('Run:', result.run_id)
    print('Directory:', result.run_dir)
    display(result.preprocessing.user_metadata.routing_eligible.value_counts(dropna=False))
    display(result.preprocessing.user_metadata.home_quality_flag.value_counts(dropna=False))
    display(result.pipeline.trip_ledger.processing_status.value_counts(dropna=False))
else:
    print('Review paths, then set EXECUTE_PRODUCTION_RUN=True.')